# Wavelet-YOLOv12 — Chen Split (Tuberculosis6208)

End-to-end Colab notebook for the Wavelet-YOLOv12 experiment.

**Setup:**
- Dataset zip on Drive: `MyDrive/Tuberculosis6208.zip` (Pascal-VOC format).
- Split: **Chen et al. IJAI 2024** — `1024 train / 140 val / 101 test`, `SPLIT_SEED=42` (deterministic).
- Models compared: `yolov12s` (baseline), `yolov12-wavelet-p3` (light), `yolov12-wavelet` (full).
- Training: end-to-end, 60 epochs, SGD lr=0.01 momentum=0.937 cos-lr, AMP, deterministic.
- Seeds: `42, 1050, 2025` (3 runs per model for variance reporting).
- Logging: **Weights & Biases** — project `wavelet_yolo12_chen`.

**Runtime:** A100 ≈ 30 min/run × 9 runs ≈ 4–5 hours.

## 1. Mount Drive & clone repo (dev/wavelet branch)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
from pathlib import Path

REPO_DIR = Path('/content/wavelet-yolo12')
BRANCH   = 'dev/wavelet'

if REPO_DIR.exists():
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull --ff-only
else:
    !git clone -b {BRANCH} https://github.com/iswantosan/wavelet-yolo12.git {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('cwd:', os.getcwd())
!git log -1 --oneline

## 2. Install dependencies

In [ ]:
# Ultralytics + W&B. We install ultralytics in editable mode so our modifications
# (WaveDown, registered modules) are picked up.
!pip -q install -e . wandb

In [ ]:
import torch, ultralytics
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('ultralytics:', ultralytics.__version__)
from ultralytics.nn.modules import WaveDown, HaarDWT
print('WaveDown registered:', WaveDown is not None)

## 3. Build Chen-style split (1024 / 140 / 101)

Pulls `Tuberculosis6208.zip` from Drive, converts VOC XML → YOLO `.txt`, deterministic shuffle with `SPLIT_SEED=42`, writes `data.yaml`. Skips work if already done unless you delete `/content/tb_chen_split/`.

In [ ]:
DRIVE_ZIP    = '/content/drive/MyDrive/Tuberculosis6208.zip'
EXTRACT_DIR  = '/content/dataset/raw'
RAW_DIR      = f'{EXTRACT_DIR}/Tuberculosis6208/tuberculosis-phonecamera'
SPLIT_DIR    = '/content/tb_chen_split'
DATA_YAML    = f'{SPLIT_DIR}/data.yaml'

!python scripts/build_chen_split.py \
    --zip "{DRIVE_ZIP}" --extract-dir "{EXTRACT_DIR}" \
    --src "{RAW_DIR}" --out "{SPLIT_DIR}"

!ls -la {SPLIT_DIR} && cat {DATA_YAML}

## 4. Smoke test — verify Wavelet-YOLOv12 builds & forwards

In [ ]:
!python scripts/smoke_test_wavelet.py

## 5. W&B login

Paste your API key from https://wandb.ai/authorize when prompted.

In [ ]:
import wandb
wandb.login()  # opens prompt; or set WANDB_API_KEY env var

## 6. Configure sweep — models × seeds

3 models × 3 seeds = **9 runs**. Reduce `SEEDS` or `MODELS` if you want to iterate quickly first.

In [ ]:
WANDB_PROJECT = 'wavelet_yolo12_chen'
RUN_PROJECT   = '/content/runs/wavelet_chen'
PRETRAINED    = 'yolov12s.pt'  # auto-downloaded by Ultralytics
EPOCHS, IMGSZ, BATCH = 60, 640, 16
SEEDS  = [42, 1050, 2025]
MODELS = [
    'ultralytics/cfg/models/v12/yolov12.yaml',           # baseline
    'ultralytics/cfg/models/v12/yolov12-wavelet-p3.yaml',# WaveDown at P3 only
    'ultralytics/cfg/models/v12/yolov12-wavelet.yaml',   # WaveDown at P3+P4+P5
]
print(f'Total runs planned: {len(MODELS) * len(SEEDS)}')

## 7. Run sweep — logs per-run to W&B, accumulates summary table

In [ ]:
import importlib, scripts.train_chen as tc
importlib.reload(tc)

results = []
for cfg in MODELS:
    for seed in SEEDS:
        print(f"\n{'='*70}\n  {cfg} | seed={seed}\n{'='*70}")
        try:
            summary = tc.train_one(
                model_cfg=cfg,
                data_yaml=DATA_YAML,
                pretrained=PRETRAINED,
                seed=seed,
                epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0,
                project=RUN_PROJECT,
                run_name=None,
                wandb_project=WANDB_PROJECT,
            )
            results.append(summary)
        except Exception as e:
            print(f'  [error] run failed: {e}')
            results.append({'model': tc.model_tag(cfg), 'seed': seed, 'error': str(e)})

## 8. Aggregate results & log a summary run to W&B

In [ ]:
import pandas as pd, numpy as np, wandb

df = pd.DataFrame(results)
print(df.to_string(index=False))
df.to_csv('/content/wavelet_chen_results.csv', index=False)

agg_rows = []
for model, sub in df.groupby('model'):
    row = {'model': model, 'n_seeds': len(sub)}
    for m in ['test_mAP50', 'test_mAP50-95', 'test_mAP@0.9', 'test_precision', 'test_recall']:
        if m in sub.columns:
            vals = pd.to_numeric(sub[m], errors='coerce').dropna()
            if len(vals):
                row[f'{m}_mean'] = float(vals.mean())
                row[f'{m}_std']  = float(vals.std(ddof=0))
    agg_rows.append(row)
agg = pd.DataFrame(agg_rows)
print('\n=== Aggregated across seeds ===')
print(agg.to_string(index=False))
agg.to_csv('/content/wavelet_chen_aggregate.csv', index=False)

sum_run = wandb.init(project=WANDB_PROJECT, name='sweep_summary', reinit=True, tags=['summary', 'chen_split'])
sum_run.log({'sweep/all_runs': wandb.Table(dataframe=df)})
sum_run.log({'sweep/aggregate': wandb.Table(dataframe=agg)})
sum_run.finish()
print('\nLogged sweep_summary to W&B.')

## 9. (Optional) Quick prediction sanity check

In [ ]:
# Pick the best run by test mAP50 and run a quick predict pass.
import pandas as pd
from ultralytics import YOLO

df = pd.read_csv('/content/wavelet_chen_results.csv')
df_ok = df.dropna(subset=['test_mAP50']) if 'test_mAP50' in df.columns else df
best  = df_ok.sort_values('test_mAP50', ascending=False).head(1)
print(best.to_string(index=False))

if len(best):
    best_pt = best.iloc[0]['best']
    m = YOLO(best_pt)
    out = m.predict('/content/tb_chen_split/test/images', save=True, imgsz=640, conf=0.25)
    print('Predict dir:', out[0].save_dir if out else None)